In [31]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn import set_config
from sklearn.ensemble import RandomForestClassifier
set_config(display='text')
loan_data = pd.read_csv('C:/medicalAI/Pycham/week3/Data_processing/decision/train_loan_80.csv')
display(loan_data.head()) 
loan_encoded = pd.get_dummies(loan_data, drop_first=True)
# display(loan_encoded.head())
x= loan_encoded.drop(columns=['Loan_Status_Y'])
y= loan_encoded['Loan_Status_Y']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,Male,Yes,0,Graduate,No,6000,2250.0,265.0,360.0,1.0,Semiurban,N
1,Male,Yes,0,Graduate,No,2958,2900.0,131.0,360.0,1.0,Semiurban,Y
2,Male,Yes,2,Graduate,No,6250,1695.0,210.0,360.0,1.0,Semiurban,Y
3,Male,Yes,0,Graduate,No,2083,3150.0,128.0,360.0,1.0,Semiurban,Y
4,Male,No,0,Graduate,No,4166,0.0,98.0,360.0,0.0,Semiurban,N


In [32]:
model = RandomForestClassifier(n_estimators=200, ## 1. Generate 200 unique decision trees 
                             max_depth=7, random_state=42,
                             max_samples= 0.7,#2. Train each tree using 70% bootstrapped data (sampling with replacement)
                             max_features= 'sqrt' ) # Randomly select sqrt(total_features) features
model.fit(x_train, y_train)
print(f'train data: {model.score(x_train, y_train)}')
print(f'test data:{model.score(x_test, y_test)}')


train data: 0.8571428571428571
test data:0.8282828282828283


In [33]:
# score importance
importances_score = model.feature_importances_
import pandas as pd

# 1. 특징 이름과 중요도 점수를 데이터프레임으로 묶기
importances = pd.Series(model.feature_importances_, index=x_train.columns)

# 2. 중요도가 높은 순서대로 내림차순 정렬하여 출력
print(importances.sort_values(ascending=False))

Credit_History             0.333936
ApplicantIncome            0.163178
LoanAmount                 0.135863
CoapplicantIncome          0.109344
Loan_Amount_Term           0.058125
Property_Area_Semiurban    0.042842
Married_Yes                0.030987
Property_Area_Urban        0.022384
Education_Not Graduate     0.022105
Dependents_1               0.018613
Gender_Male                0.017464
Self_Employed_Yes          0.016122
Dependents_2               0.015182
Dependents_3+              0.013854
dtype: float64


In [34]:
#cross-validation, k-fold
import numpy as np, pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)#k=5
tree_cv = cross_val_score(DecisionTreeClassifier(max_depth=7, random_state=1), x, y, cv=cv)
print(f'decesion Redd 5-fold score:', np.round(tree_cv, 3))
print('mean:', round(tree_cv.mean(), 3))

decesion Redd 5-fold score: [0.778 0.704 0.786 0.755 0.847]
mean: 0.774


In [35]:
#random Forest (Majority Vote)
from sklearn.ensemble import RandomForestClassifier
rf_cv = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=1), x, y, cv=cv)
print(f'random Forest 5-fold score:', np.round(rf_cv, 3))
print(f'mean: {round(rf_cv.mean(), 3)}')

random Forest 5-fold score: [0.818 0.765 0.796 0.806 0.827]
mean: 0.802


In [37]:
#hyper_parameter tuning = GridSearchCV
# Test all hyperparameter combinations using cross-validation to find the best configuration
from sklearn.model_selection import GridSearchCV
rf_grid = GridSearchCV(RandomForestClassifier(random_state=1),
                       {'n_estimators': [100, 200, 300, 400], 'max_depth':[4, 6, 8, None],
                       'min_samples_leaf': [1, 3, 5, 7]}, cv=cv, scoring='accuracy').fit(x,y)
print('random forest tuning max:', round(rf_grid.best_score_, 3), rf_grid.best_params_)

random forest tuning max: 0.815 {'max_depth': 4, 'min_samples_leaf': 5, 'n_estimators': 100}


In [26]:
#gradient Boosting: Sequentially builds trees with a focus on residual errors
from sklearn.ensemble import GradientBoostingClassifier
gb_cv = cross_val_score(GradientBoostingClassifier(random_state=1), x, y, cv=cv)
print(f'gradient boostin mean:{round(gb_cv.mean(), 3)}')

gradient boostin mean:0.786


In [27]:
from sklearn.ensemble import GradientBoostingClassifier

# 1. 그래디언트 부스팅 기본 교차 검증 (k-fold=5)
gb_cv = cross_val_score(GradientBoostingClassifier(random_state=1), x, y, cv=cv)
print(f'Gradient Boosting 5-fold score mean:', round(gb_cv.mean(), 3))

# 2. 그래디언트 부스팅도 하이퍼파라미터 튜닝(GridSearchCV) 가능!
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=1),
    {
        'n_estimators': [100, 200], 
        'learning_rate': [0.01, 0.05, 0.1], # 부스팅만의 핵심 파라미터 (학습률)
        'max_depth': [3, 5]
    }, 
    cv=cv, 
    scoring='accuracy'
).fit(x, y)

print('Gradient Boosting tuning max:', round(gb_grid.best_score_, 3), gb_grid.best_params_)

Gradient Boosting 5-fold score mean: 0.786
Gradient Boosting tuning max: 0.804 {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
